In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime
import xml.etree.ElementTree as ET
from xml.dom import minidom

# ─── CONFIGURATION ────────────────────────────────────────────────────────────
SAM_OUTPUT_DIR = "./sam_output"     # output from Script 1
CVAT_OUTPUT    = "./cvat_import/annotations.xml"
DUMMY_LABEL    = "rock"             # assigned to ALL instances
LABEL_COLOR    = "#FF6600"          # display colour in CVAT (optional)

# ─── TASK METADATA ────────────────────────────────────────────────────────────
# These values must match the CVAT task you will import into.
# If unsure: use placeholder values — CVAT ignores most meta fields on import.
TASK_NAME      = "Rock Segmentation"
TASK_ID        = "1"

os.makedirs(Path(CVAT_OUTPUT).parent, exist_ok=True)

In [ ]:
def points_to_cvat_string(flat_points: list) -> str:
    """
    Convert flat list [x1, y1, x2, y2, ...] to CVAT polygon points string:
    "x1.00,y1.00;x2.00,y2.00;..."
    """
    pairs = []
    it = iter(flat_points)
    for x in it:
        y = next(it)
        pairs.append(f"{float(x):.2f},{float(y):.2f}")
    return ";".join(pairs)


def build_cvat_xml(
    sam_output_dir : str,
    dummy_label    : str,
    label_color    : str,
    task_name      : str,
    task_id        : str
) -> ET.Element:
    """
    Build a CVAT XML 1.1 annotation tree from per-image JSON files.

    The CVAT XML 1.1 schema:
      <annotations>
        <version>1.1</version>
        <meta> ... task info ... </meta>
        <image id="N" name="filename.jpg" width="W" height="H">
          <polygon label="rock" points="..." z_order="0" occluded="0"/>
          ...
        </image>
      </annotations>
    """
    json_files = sorted(Path(sam_output_dir).glob("*.json"))
    if not json_files:
        raise FileNotFoundError(f"No JSON files found in {sam_output_dir}")

    print(f"Found {len(json_files)} JSON files to convert.")

    # ─ Root element ──────────────────────────────────────────────────────────
    root = ET.Element("annotations")
    ET.SubElement(root, "version").text = "1.1"

    # ─ Meta block ────────────────────────────────────────────────────────────
    meta  = ET.SubElement(root, "meta")
    task  = ET.SubElement(meta, "task")
    ET.SubElement(task, "id").text   = task_id
    ET.SubElement(task, "name").text = task_name
    ET.SubElement(task, "size").text = str(len(json_files))
    ET.SubElement(task, "mode").text = "annotation"

    segments = ET.SubElement(task, "segments")
    seg      = ET.SubElement(segments, "segment")
    ET.SubElement(seg, "id").text    = "1"
    ET.SubElement(seg, "start").text = "0"
    ET.SubElement(seg, "stop").text  = str(len(json_files) - 1)

    labels_el = ET.SubElement(task, "labels")
    label_el  = ET.SubElement(labels_el, "label")
    ET.SubElement(label_el, "name").text  = dummy_label
    ET.SubElement(label_el, "color").text = label_color
    ET.SubElement(label_el, "attributes")

    ET.SubElement(meta, "dumped").text = datetime.now().isoformat()

    # ─ Image blocks ──────────────────────────────────────────────────────────
    total_polygons = 0
    skipped_images = 0

    for img_idx, json_path in enumerate(json_files):
        with open(json_path) as f:
            record = json.load(f)

        if record["n_masks"] == 0:
            skipped_images += 1
            continue

        img_el = ET.SubElement(root, "image")
        img_el.set("id",     str(img_idx))
        img_el.set("name",   record["filename"])   # MUST match CVAT task filename
        img_el.set("width",  str(record["width"]))
        img_el.set("height", str(record["height"]))

        for mask in record["masks"]:
            for poly_pts in mask["polygons"]:
                if len(poly_pts) < 6:  # need at least 3 points (x,y pairs)
                    continue

                poly_el = ET.SubElement(img_el, "polygon")
                poly_el.set("label",    dummy_label)
                poly_el.set("points",   points_to_cvat_string(poly_pts))
                poly_el.set("z_order",  "0")
                poly_el.set("occluded", "0")
                total_polygons += 1

    print(f"\nSummary:")
    print(f"  Images with annotations : {len(json_files) - skipped_images}")
    print(f"  Images with 0 masks     : {skipped_images}")
    print(f"  Total polygons written  : {total_polygons}")

    return root


def write_pretty_xml(element: ET.Element, output_path: str):
    """Write XML with indentation (human-readable)."""
    raw    = ET.tostring(element, encoding="unicode")
    parsed = minidom.parseString(raw)
    pretty = parsed.toprettyxml(indent="  ", encoding="utf-8")

    with open(output_path, "wb") as f:
        f.write(pretty)
    print(f"\nAnnotation file saved to: {output_path}")
    print(f"File size: {Path(output_path).stat().st_size / 1024:.1f} KB")


# ─── RUN ──────────────────────────────────────────────────────────────────────
root = build_cvat_xml(
    sam_output_dir = SAM_OUTPUT_DIR,
    dummy_label    = DUMMY_LABEL,
    label_color    = LABEL_COLOR,
    task_name      = TASK_NAME,
    task_id        = TASK_ID
)

write_pretty_xml(root, CVAT_OUTPUT)

In [ ]:
def validate_cvat_xml(xml_path: str):
    """Quick structural validation of the generated XML."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    version  = root.findtext("version")
    images   = root.findall("image")
    polygons = root.findall(".//polygon")
    labels   = {el.findtext("name") for el in root.findall(".//label")}

    print(f"CVAT XML Validation")
    print(f"  Version   : {version}")
    print(f"  Images    : {len(images)}")
    print(f"  Polygons  : {len(polygons)}")
    print(f"  Labels    : {labels}")

    # Check for common issues
    issues = []
    for img in images:
        if not img.get("name"):
            issues.append(f"  [WARN] <image> missing 'name' attribute: id={img.get('id')}")
        polys = img.findall("polygon")
        for p in polys:
            pts = p.get("points", "")
            n_coords = len(pts.split(";"))
            if n_coords < 3:
                issues.append(
                    f"  [WARN] Polygon with < 3 points in {img.get('name')}"
                )

    if issues:
        print("\nIssues found:")
        for issue in issues:
            print(issue)
    else:
        print("\n  No issues found. File is ready for CVAT import.")

validate_cvat_xml(CVAT_OUTPUT)